# LoRA 服务

SGLang 支持在基础模型上使用 [LoRA 适配器](https://arxiv.org/abs/2106.09685)。通过结合 [S-LoRA](https://arxiv.org/pdf/2311.03285) 和 [Punica](https://arxiv.org/pdf/2310.18547) 的技术，SGLang 可以在单个输入批次中为不同序列高效支持多个 LoRA 适配器。

## LoRA 服务的参数

以下服务器参数与多 LoRA 服务相关：

* `enable_lora`：启用模型的 LoRA 支持。如果提供了 `--lora-paths`，为了向后兼容，此参数会自动设置为 True。

* `enable_lora_overlap_loading`：启用异步 LoRA 权重加载，以便将 H2D 传输与 GPU 计算重叠。如果您发现 LoRA 工作负载因适配器权重加载而成为瓶颈（例如频繁加载大型 LoRA 适配器），应启用此选项。

* `lora_paths`：要加载的 LoRA 适配器列表。每个适配器必须以以下格式之一指定：<PATH> | <NAME>=<PATH> | 带有 {"lora_name":str,"lora_path":str,"pinned":bool} 模式的 JSON。

* `max_loras_per_batch`：每个批次使用的最大适配器数量。此参数会影响为多 LoRA 服务预留的 GPU 内存量，因此在内存不足时应设置为较小的值。默认为 8。

* `max_loaded_loras`：如果指定，限制同一时间加载到 CPU 内存中的最大 LoRA 适配器数量。该值必须大于或等于 `max-loras-per-batch`。

* `lora_eviction_policy`：GPU 内存池满时的 LoRA 适配器驱逐策略。`lru`：最近最少使用（默认，缓存效率更好）。`fifo`：先进先出。

* `lora_backend`：运行 LoRA 模块 GEMM 内核的后端。目前支持 Triton LoRA 后端（`triton`）和 Chunked SGMV 后端（`csgmv`）。未来将添加基于 Cutlass 或 CUDA 内核的更快后端。

* `max_lora_rank`：应支持的最大 LoRA 秩。如果未指定，将从 `--lora-paths` 提供的适配器中自动推断。当您期望在服务器启动后动态加载更大 LoRA 秩的适配器时，需要此参数。

* `lora_target_modules`：所有应应用 LoRA 的目标模块的并集（例如 `q_proj`、`k_proj`、`gate_proj`）。如果未指定，将从 `--lora-paths` 提供的适配器中自动推断。当您期望在服务器启动后动态加载不同目标模块的适配器时，需要此参数。您也可以将其设置为 `all` 以对所有支持的模块启用 LoRA。但是，在额外模块上启用 LoRA 会引入少量性能开销。如果您的应用对性能敏感，建议仅指定计划加载适配器的模块。

* `--max-lora-chunk-size`：ChunkedSGMV LoRA 后端的最大块大小。仅在 --lora-backend 为 'csgmv' 时使用。选择更大的值可能提高性能。请根据您的硬件和工作负载调整此值。默认为 16。

* `tp_size`：SGLang 支持 LoRA 服务与张量并行一起使用。`tp_size` 控制用于张量并行的 GPU 数量。有关张量分片策略的更多详情，请参见 [S-Lora](https://arxiv.org/pdf/2311.03285) 论文。

从客户端来看，用户需要提供一个字符串列表作为输入批次，以及每个输入序列对应的适配器名称列表。

## 用法

### 服务单个适配器

**注意：** SGLang 通过两种 API 支持 LoRA 适配器：

1. **OpenAI 兼容 API**（`/v1/chat/completions`、`/v1/completions`）：使用 `model:adapter-name` 语法。示例请参阅[使用 LoRA 的 OpenAI API](../basic_usage/openai_api_completions.ipynb#Using-LoRA-Adapters)。

2. **原生 API**（`/generate`）：在请求体中传递 `lora_path`（如下所示）。

In [ ]:
import json
import requests

from sglang.test.doc_patch import launch_server_cmd
from sglang.utils import wait_for_server, terminate_process

In [ ]:
server_process, port = launch_server_cmd(
    # Here we set max-loras-per-batch to 2: one slot for adaptor and another one for base model
    """
python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-lora \
    --lora-paths lora0=algoprog/fact-generation-llama-3.1-8b-instruct-lora \
    --max-loras-per-batch 2 \
    --log-level warning \
"""
)

wait_for_server(f"http://localhost:{port}", process=server_process)

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses lora0, and the second input uses the base model
    "lora_path": ["lora0", None],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output 0: {response.json()[0]['text']}")
print(f"Output 1: {response.json()[1]['text']}")

In [ ]:
terminate_process(server_process)

### 服务多个适配器

In [ ]:
server_process, port = launch_server_cmd("""
python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-lora \
    --lora-paths lora0=algoprog/fact-generation-llama-3.1-8b-instruct-lora \
    lora1=Nutanix/Meta-Llama-3.1-8B-Instruct_lora_4_alpha_16 \
    --max-loras-per-batch 2 \
    --log-level warning \
""")

wait_for_server(f"http://localhost:{port}", process=server_process)

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses lora0, and the second input uses lora1
    "lora_path": ["lora0", "lora1"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output 0: {response.json()[0]['text']}")
print(f"Output 1: {response.json()[1]['text']}")

In [ ]:
terminate_process(server_process)

### 动态 LoRA 加载

您也可以通过 `/load_lora_adapter` 和 `/unload_lora_adapter` API 动态加载和卸载 LoRA 适配器，而无需在服务器启动时通过 `--lora-paths` 指定所有适配器。

使用动态 LoRA 加载时，建议在启动时显式指定 `--max-lora-rank` 和 `--lora-target-modules`。为了向后兼容，如果未显式提供这些值，SGLang 会从 `--lora-paths` 推断。但在这种情况下，您需要确保所有动态加载的适配器与初始 `--lora-paths` 中的适配器具有相同的形状（秩和目标模块），或者严格"更小"。

In [ ]:
lora0 = "Nutanix/Meta-Llama-3.1-8B-Instruct_lora_4_alpha_16"  # rank - 4, target modules - q_proj, k_proj, v_proj, o_proj, gate_proj
lora1 = "algoprog/fact-generation-llama-3.1-8b-instruct-lora"  # rank - 64, target modules - q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
lora0_new = "philschmid/code-llama-3-1-8b-text-to-sql-lora"  # rank - 256, target modules - q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj


# The `--target-lora-modules` param below is technically not needed, as the server will infer it from lora0 which already has all the target modules specified.
# We are adding it here just to demonstrate usage.
server_process, port = launch_server_cmd("""
    python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-lora \
    --cuda-graph-max-bs 2 \
    --max-loras-per-batch 2 \
    --max-lora-rank 256
    --lora-target-modules all
    --log-level warning
    """)

url = f"http://127.0.0.1:{port}"
wait_for_server(url, process=server_process)

加载适配器 lora0

In [ ]:
response = requests.post(
    url + "/load_lora_adapter",
    json={
        "lora_name": "lora0",
        "lora_path": lora0,
    },
)

if response.status_code == 200:
    print("LoRA adapter loaded successfully.", response.json())
else:
    print("Failed to load LoRA adapter.", response.json())

加载适配器 lora1：

In [ ]:
response = requests.post(
    url + "/load_lora_adapter",
    json={
        "lora_name": "lora1",
        "lora_path": lora1,
    },
)

if response.status_code == 200:
    print("LoRA adapter loaded successfully.", response.json())
else:
    print("Failed to load LoRA adapter.", response.json())

检查推理输出：

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses lora0, and the second input uses lora1
    "lora_path": ["lora0", "lora1"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output from lora0: \n{response.json()[0]['text']}\n")
print(f"Output from lora1 (updated): \n{response.json()[1]['text']}\n")

卸载 lora0 并替换为不同的适配器：

In [ ]:
response = requests.post(
    url + "/unload_lora_adapter",
    json={
        "lora_name": "lora0",
    },
)

response = requests.post(
    url + "/load_lora_adapter",
    json={
        "lora_name": "lora0",
        "lora_path": lora0_new,
    },
)

if response.status_code == 200:
    print("LoRA adapter loaded successfully.", response.json())
else:
    print("Failed to load LoRA adapter.", response.json())

再次检查输出：

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses lora0, and the second input uses lora1
    "lora_path": ["lora0", "lora1"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output from lora0: \n{response.json()[0]['text']}\n")
print(f"Output from lora1 (updated): \n{response.json()[1]['text']}\n")

In [ ]:
terminate_process(server_process)

### OpenAI 兼容 API 用法

您可以通过 OpenAI 兼容 API 使用 LoRA 适配器，在 `model` 字段中使用 `base-model:adapter-name` 语法（例如 `qwen/qwen2.5-0.5b-instruct:adapter_a`）指定适配器。更多详情和示例，请参阅 OpenAI API 文档中的"使用 LoRA 适配器"部分：[openai_api_completions.ipynb](../basic_usage/openai_api_completions.ipynb)。


### LoRA GPU 固定

另一个高级选项是在加载时将适配器指定为 `pinned`。当适配器被固定时，它会永久分配到可用的 GPU 池槽位之一（由 `--max-loras-per-batch` 配置），并且在运行时不会从 GPU 内存中被驱逐。相反，它会一直驻留直到被显式卸载。

这可以在同一适配器频繁被请求使用的场景中提高性能，避免重复的内存传输和重初始化开销。但是，由于 GPU 池槽位有限，固定适配器会降低系统动态按需加载其他适配器的灵活性。如果固定的适配器过多，可能导致性能下降，或在最极端的情况下（`固定适配器数量 == max-loras-per-batch`）使所有未固定的请求停滞。因此，SGLang 目前将最大固定适配器数量限制为 `max-loras-per-batch - 1`，以防止意外的饥饿。

在以下示例中，我们启动的服务器将 `lora1` 作为固定适配器加载，`lora2` 和 `lora3` 作为常规（未固定）适配器加载。请注意，我们故意使用两种不同的格式指定 `lora2` 和 `lora3`，以展示两种格式都被支持。

In [ ]:
server_process, port = launch_server_cmd("""
    python3 -m sglang.launch_server --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-lora \
    --cuda-graph-max-bs 8 \
    --max-loras-per-batch 3 \
    --max-lora-rank 256 \
    --lora-target-modules all \
    --lora-paths \
        {"lora_name":"lora0","lora_path":"Nutanix/Meta-Llama-3.1-8B-Instruct_lora_4_alpha_16","pinned":true} \
        {"lora_name":"lora1","lora_path":"algoprog/fact-generation-llama-3.1-8b-instruct-lora"} \
        lora2=philschmid/code-llama-3-1-8b-text-to-sql-lora
    --log-level warning
    """)


url = f"http://127.0.0.1:{port}"
wait_for_server(url, process=server_process)

您还可以在动态适配器加载期间将适配器指定为固定。在以下示例中，我们将 `lora2` 重新加载为固定适配器：

In [ ]:
response = requests.post(
    url + "/unload_lora_adapter",
    json={
        "lora_name": "lora1",
    },
)

response = requests.post(
    url + "/load_lora_adapter",
    json={
        "lora_name": "lora1",
        "lora_path": "algoprog/fact-generation-llama-3.1-8b-instruct-lora",
        "pinned": True,  # Pin the adapter to GPU
    },
)

验证结果是否符合预期：

In [ ]:
url = f"http://127.0.0.1:{port}"
json_data = {
    "text": [
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": {"max_new_tokens": 32, "temperature": 0},
    # The first input uses lora0, and the second input uses lora1
    "lora_path": ["lora0", "lora1", "lora2"],
}
response = requests.post(
    url + "/generate",
    json=json_data,
)
print(f"Output from lora0 (pinned): \n{response.json()[0]['text']}\n")
print(f"Output from lora1 (pinned): \n{response.json()[1]['text']}\n")
print(f"Output from lora2 (not pinned): \n{response.json()[2]['text']}\n")

In [ ]:
terminate_process(server_process)

## 选择 LoRA 后端

SGLang 支持两种 LoRA 后端，您可以使用 `--lora-backend` 参数进行选择：

- `triton`：基本的 Triton 后端。
- `csgmv`：默认的 chunked SGMV 后端，针对高并发场景优化。

`csgmv` 后端是最近引入的，旨在提高特别是在高并发场景下的性能。我们的基准测试表明，与基本的 triton 后端相比，它实现了 20% 到 80% 的延迟改善。

In [ ]:
server_process, port = launch_server_cmd("""
    python3 -m sglang.launch_server \
    --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-lora \
    --lora-backend csgmv \
    --max-loras-per-batch 16 \
    --lora-paths lora1=path/to/lora1 lora2=path/to/lora2
    """)

In [ ]:
terminate_process(server_process)

## LoRA 重叠加载

通过使用 `--enable-lora-overlap-loading` 服务器参数，SGLang 引擎能够将 LoRA 权重的加载与预填充和解码计算重叠，从而将 LoRA 权重的数据移动隐藏在 GPU 计算之后。我们的基准测试表明，在对抗性条件下，启用此特性可以使中位 TTFT 降低约 35%——（详细基准请参见 [LoRA 重叠加载 PR](https://github.com/sgl-project/sglang/pull/15512)）。

In [ ]:
lora0 = "Nutanix/Meta-Llama-3.1-8B-Instruct_lora_4_alpha_16"
lora1 = "algoprog/fact-generation-llama-3.1-8b-instruct-lora"
lora2 = "philschmid/code-llama-3-1-8b-text-to-sql-lora"


server_process, port = launch_server_cmd("""
    python3 -m sglang.launch_server \
    --model-path meta-llama/Meta-Llama-3.1-8B-Instruct \
    --enable-lora \
    --enable-lora-overlap-loading \
    --lora-paths lora0=Nutanix/Meta-Llama-3.1-8B-Instruct_lora_4_alpha_16 \
    lora1=algoprog/fact-generation-llama-3.1-8b-instruct-lora \
    lora2=philschmid/code-llama-3-1-8b-text-to-sql-lora \
    --max-lora-rank 256 \
    --max-loras-per-batch 2 \
    --max-loaded-loras 4
    """)

url = f"http://127.0.0.1:{port}"
wait_for_server(url, process=server_process)

In [ ]:
json_data = {
    "text": [
        "Write a very long fairy-tale.",
        "List 3 countries and their capitals.",
        "List 3 countries and their capitals.",
    ],
    "sampling_params": [
        {"max_new_tokens": 1024, "temperature": 0},
        {"max_new_tokens": 64, "temperature": 0},
        {"max_new_tokens": 64, "temperature": 0},
    ],
    "lora_path": ["lora0", "lora1", "lora2"],
}

# lora0 and lora1 will be loaded into the memory pool first, and because max_loras_per_batch = 2, lora2's request will remain in the queue.
# lora1's request will likely finish first, and once it does, lora2 will be loaded. With --enable-lora-overlap-loading, this loading will
# occur asynchronously and thus decoding for lora0's request won't be blocked.
response = requests.post(
    url + "/generate",
    json=json_data,
)

for i in range(3):
    print(f"Output from lora{i}: \n{response.json()[i]['text']}\n")

In [ ]:
terminate_process(server_process)

#### LoRA 重叠加载的限制

然而，LoRA 重叠加载并非没有代价，它有两个重要的注意事项：

1. **固定 CPU 内存需求**：
   异步 H2D 内存复制需要将 LoRA 权重固定在 CPU 内存中，而这是有限的系统资源。为了缓解过度使用固定内存的问题，SGLang 目前在启用 LoRA 重叠加载时将 `max_loaded_loras` 限制为最多 `max_loras_per_batch` 的 2 倍。

2. **减少多适配器预填充批处理**：
   使用重叠加载时，适配器在不同时间可用于 GPU，因为每个适配器是异步加载的。这可能降低调度器形成多适配器预填充批次的能力，因为只有适配器当前已加载的请求才能被分组在一起。因此，不同适配器的请求将被安排在单独的（或更小的）预填充批次中，当适配器加载时间相对于预填充计算时间较小时，这可能增加 TTFT。这就是为什么 LoRA 重叠加载默认禁用的原因：仅当用户确定 LoRA 权重加载是瓶颈时（例如高适配器变动、大量适配器权重或 PCIe 瓶颈工作负载）才应启用。


#### 重叠加载导致更高延迟的示例

例如，假设我们有四个 LoRA 适配器：`lora0`、`lora1`、`lora2` 和 `lora3`。加载任何适配器需要 2ms，而该适配器的预填充步骤需要 20ms。

1. **基线**：
  引擎同步加载所有四个适配器，然后运行一个组合的预填充批次，总时间约为 `2 * 4 + 20 = 28ms`

2. **启用 LoRA 重叠加载**：
  引擎开始加载 `lora0`，一旦就绪，就安排一个仅包含 `lora0` 的预填充批次，同时在后台加载 `lora1`。然后在 `lora1` 加载时安排其预填充，以此类推。在最坏情况下，即预填充无法跨适配器批处理时，总时间约为 `2 + 4 * 20 = 82ms`

在这种场景中，重叠加载减少了适配器加载开销，但多适配器预填充批处理的损失占主导地位，导致更高的 TTFT。

## 未来工作

LoRA 相关功能的开发路线图可在此 [issue](https://github.com/sgl-project/sglang/issues/2929) 中找到。其他功能，包括嵌入层、统一分页、Cutlass 后端等仍在开发中。